In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.linalg import cholesky
from numpy.random import multivariate_normal
from scipy.optimize import curve_fit
from pathlib import Path

from scipy.interpolate import interp1d
from scipy.integrate import cumulative_trapezoid
from scipy import interpolate

import os
import seaborn as sns
import pandas as pd
import importlib

#Plot param
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset']='dejavuserif'
plt.rcParams["figure.figsize"] = [15,10]
plt.rcParams["font.size"] = 10

In [2]:
import gp_sampler
from gp_sampler import GPSampler

import extrapolation_hrg as extrap_hrg
import extrapolation_powerlaw as extrap_pwrl
import extrapolation_weighted as extrap_w

In [3]:
#units

MeV_to_fm_m1 = 1/197.3
GeV_to_fm_m1 = 5.068
MeV_to_GeV = 1e-3
hbarc = 197.3 # MeV fm
MeV4_to_GeV_fm3 = (1e-3) * (1 / hbarc**3)

In [4]:
#temperature vectors with stepsize one
T_max = 900

T_coarse = np.linspace(120,260,260-120+1)
T_total_coarse = np.linspace(120,T_max,T_max-120+1)
T_up_coarse = np.linspace(261,T_max,T_max-261+1)

#ranges in temperature with smaller stepsize

step_down = 0.001
step_up = 0.01

T_point01_up = np.linspace(120, T_max, int((T_max - 120) / step_up) + 1)
T_point001_down = np.round(np.arange(0, 119.9 + step_down, step_down), 10)
T_total = np.concatenate((T_point001_down, T_point01_up))

T_point001_110 = T_point001_down[0:int(110/step_down)]
T_concatenate_110to120 = np.concatenate([T_point001_110, T_point01_up])

In [5]:
#switching temperature and indices

T_sw = 136 #MeV

idx_Tsw = int(len(T_point001_down) + (T_sw-120)/step_up)
idx_Tsw_up = int((T_sw-120)/step_up)

In [6]:
#temperature indices for plotting and testing

idx_30 = int(30/step_down)
idx_40 = int(40/step_down)
idx_90 = int(90/step_down)
idx_100 = int(100/step_down)
idx_110 = int(110/step_down)
idx_120 = len(T_point001_down) 
idx_130 = int(len(T_point001_down) + 10/step_up)
idx_140 = int(len(T_point001_down) + 20/step_up)
idx_150 = int(len(T_point001_down) + 30/step_up)
idx_160 = int(len(T_point001_down) + 40/step_up)
idx_170 = int(len(T_point001_down) + 50/step_up)

len_transition = len(T_total[0:idx_120]) - len(T_total[0:idx_110])
len_transition_unc = len(T_total[0:idx_120]) - len(T_total[0:idx_90])

## Reading in lattice data

In [7]:
data = pd.read_csv("./data/cs2_WB.dat",sep=r"\s+",header=None)

#data extracted from graph up to ~400 MeV and taken as 1/3 at 700 MeV
data_extracted = pd.read_csv("./data/cs2_data_extrap/points_extracted",sep=",",header=None)

In [8]:
lattice_cs2 = data[[0,1,2]]
extracted_cs2 = data_extracted[[0,1]]

In [9]:
lattice_cs2.columns = ['T','cs2','error']
extracted_cs2.columns = ['T_extract', 'cs2_extract']

In [10]:
lattice_cs2.to_csv("./data/data.dat", sep= ' ', index = False)
extracted_cs2.to_csv("./data/data_extracted.dat", sep= ',', index = False)

extracted_cs2_interp = interp1d(extracted_cs2['T_extract'], extracted_cs2['cs2_extract'], kind='linear') 

In [11]:
mean_data = np.concatenate([lattice_cs2['cs2'], extracted_cs2_interp(T_up_coarse)])

## HRG

### Using UrQMD particle list

In [12]:
#read particle list from file

file_path = "./data/pdg-urqmd_v3.3+.dat"

if not os.path.isfile(file_path):
    print("Unable to open file")
    exit(1)

m_bosons = []
m_fermions = []
g_b = []
g_f = []

with open(file_path, "r") as file:
    for line in file:
        
        #removes whitespace characters, including spaces, tabs, and newlines
        line = line.strip() 
        
        if not line:
            continue  # skip empty lines

        values = line.split()

        if values[1][0].isalpha():
            try:
                mass = float(values[2])
                f_or_b = int(values[5])
                g = float(values[4])
                mass_mev = mass * 1e3  #MeV

                if f_or_b == 1:
                    m_fermions.append(mass_mev)
                    g_f.append(g)
                elif f_or_b == 0:
                    m_bosons.append(mass_mev)
                    g_b.append(g)
            except ValueError:
                continue  
        elif values[1][0].isdigit():
            continue  


In [13]:
import hrg
from hrg import HRGCalculator

hrg = HRGCalculator(m_fermions, g_f, m_bosons, g_b)

#saving HRG quantities to file to use later, in the range of temperatures up to 170 MeV
#only run the next line again if bigger HRG temp range needed

# hrg.save_thermo_table(T_total[:idx_170], "hrg_thermo_table.txt")

In [14]:
P_hrg_total = np.loadtxt("./data/hrg_thermo_table.txt", usecols=0, skiprows=1)
e_hrg_total = np.loadtxt("./data/hrg_thermo_table.txt", usecols=1, skiprows=1)
s_hrg_total = np.loadtxt("./data/hrg_thermo_table.txt", usecols=2, skiprows=1)

In [15]:
#T_sw can be changed above

P_sw = P_hrg_total[idx_Tsw]
e_sw = e_hrg_total[idx_Tsw]
s_sw = s_hrg_total[idx_Tsw]

## Thermodynamics from sample

In [16]:
def s_integrand(T, cs2_interp): 
    return 1 / (T * cs2_interp(T))

def entropy_up(s_sw, T_sw, cs2_interp, T_up):

    integrand = s_integrand(T_up, cs2_interp)
    integral = cumulative_trapezoid(integrand, T_up, initial=0)

    Int_at_Tsw = np.interp(T_sw, T_up, integral)

    #subtracts Int_at_Tsw from the integral, s(T_sw) = s_sw
    integral -= Int_at_Tsw

    s_up_vec = s_sw * np.exp(integral)
    s_interp = interp1d(T_up, s_up_vec)

    return s_interp

In [17]:
def P_integrand(T, s_interp):
    return s_interp(T)

def pressure_up(P_sw, T_sw, T_vec, s_interp):
    T_vec = np.asarray(T_vec)

    integrand = s_interp(T_vec)
    integral = cumulative_trapezoid(integrand, T_vec, initial=0)

    # find switch index
    i_sw = np.argmin(np.abs(T_vec - T_sw))

    # shift so that P(T_sw) = P_sw
    pressure = P_sw + integral - integral[i_sw]

    P_interp = interp1d(T_vec, pressure)

    return P_interp

In [18]:
#energy upwards

def energy_up(s, P, T_vec):
    T_vec = np.asarray(T_vec, dtype=float)
    e_vals = np.fromiter(
        (s(T) * T - P(T) for T in T_vec),
        dtype=float,
        count=T_vec.size,
    )
    return lambda x: np.interp(np.asarray(x, dtype=float), T_vec, e_vals)

In [19]:
#HRG vectors low T

P_hrg_to110 = P_hrg_total[:idx_110]
e_hrg_to110 = e_hrg_total[:idx_110]
s_hrg_to110 =[]
for i in range(0, len(e_hrg_to110)):
    s_hrg_to110.append(np.where(T_total[i] != 0, (e_hrg_to110[i] + P_hrg_to110[i]) / T_total[i], 0.0))

/tmp/ipykernel_1160564/3349520596.py:7: RuntimeWarning: invalid value encountered in scalar divide
  s_hrg_to110.append(np.where(T_total[i] != 0, (e_hrg_to110[i] + P_hrg_to110[i]) / T_total[i], 0.0))


In [20]:
P_hrgtoTsw = P_hrg_total[:idx_Tsw]
e_hrgtoTsw = e_hrg_total[:idx_Tsw]
s_hrgtoTsw = s_hrg_total[:idx_Tsw]

## Saving to file (constrained gp)

In [21]:
n_samples = 2
l = 400
sigma = 15

In [22]:
sampler = GPSampler(lattice_cs2, extracted_cs2, T_up_coarse, T_total_coarse)

In [23]:
output_dir = Path(f"./eosDataFiles/Tsw_{T_sw}/constrained/l{l}_s{sigma}")
output_dir.mkdir(parents=True, exist_ok=True)

In [24]:
def make_monotonic_eos(T, e, p, cs2, s, tol=1e-8):
    """Keep rows with strictly increasing energy density."""
    T = np.asarray(T)
    e = np.asarray(e)
    p = np.asarray(p)
    cs2 = np.asarray(cs2)
    s = np.asarray(s)

    keep = [0]
    last_e = e[0]

    for i in range(1, len(e)):
        if np.isfinite(e[i]) and e[i] > last_e + tol:
            keep.append(i)
            last_e = e[i]

    keep = np.asarray(keep, dtype=int)
    return T[keep], e[keep], p[keep], cs2[keep], s[keep]

In [25]:

def save_constrained_gp_files(sample_idx, l, sigma):
    while True:
        cs2_sample_file = sampler.generate_sample(l=10, sigma=15)
        cs2_interp_file = interp1d(T_total_coarse, cs2_sample_file) 

        s_up_file = entropy_up(s_sw, T_sw, cs2_interp_file, T_point01_up)
        P_up_file = pressure_up(P_sw, T_sw, T_point01_up, s_up_file)
        e_up_file = energy_up(s_up_file, P_up_file, T_point01_up)

        #pwr extrap 
        thermo_pwrl_file = extrap_pwrl.extrapolate_powerlaw(
          T_down=T_point001_down, 
          T_up=T_point01_up, 
          T_total=T_total, 
          cs2_interp=cs2_interp_file, 
          s_up_interp=s_up_file, 
          P_up_interp=P_up_file, 
        )

        P_pwrl_file = thermo_pwrl_file['pressure']
        e_pwrl_file = thermo_pwrl_file['energy'] 
        s_pwrl_file = thermo_pwrl_file['entropy']
        cs2_pwrl_file = thermo_pwrl_file['cs2']

        #weighted extrap 
        thermo_w_file = extrap_w.extrapolate_weighted(
          P_down=P_hrg_to110,
          P_up = P_up_file(T_point01_up),
          e_down=e_hrg_to110,
          e_up = e_up_file(T_point01_up),
          T_up = T_point01_up,
          T_total=T_total, 
          cs2_interp=cs2_interp_file, 
          n_transition=len_transition,
          endpointIdx = idx_120
        )    

        P_w_file = thermo_w_file['pressure']
        e_w_file = thermo_w_file['energy'] 
        s_w_file = thermo_w_file['entropy']
        cs2_w_file = thermo_w_file['cs2']
        adjusted_transition_P = thermo_w_file["adjusted_transition_P"]
        adjusted_transition_e = thermo_w_file["adjusted_transition_e"]

        cs2_region = thermo_w_file['cs2'][idx_110:idx_120]

        if adjusted_transition_P != adjusted_transition_e:
           print("Rejected sample: transition width adjusted")
           continue

        if np.any((cs2_region < 0) | (cs2_region > 1.0001)):
           print("Rejected sample: cs2 out of bounds in transition region")
           continue

        #hrg to 150 vectors
        thermo_hrg_file = extrap_hrg.extrapolate_hrg(
          P_down=P_hrgtoTsw,
          P_up = P_up_file,
          e_down=e_hrgtoTsw,
          e_up=e_up_file,
          s_down=s_hrgtoTsw,
          s_up=s_up_file,
          T_total = T_total,
          cs2_interp=cs2_interp_file,
          idx_Tsw = idx_Tsw,
        )   

        P_hrg_file = thermo_hrg_file['pressure']
        e_hrg_file = thermo_hrg_file['energy'] 
        s_hrg_file = thermo_hrg_file['entropy']
        cs2_hrg_file = thermo_hrg_file['cs2']

        T_pwr, e_pwr, P_pwr, cs2_pwr, s_pwr = make_monotonic_eos(
            T_total, e_pwrl_file, P_pwrl_file, cs2_pwrl_file, s_pwrl_file
        )
        with open(output_dir / f"eos_pwr_s{sample_idx}_l{l}_s{sigma}.dat", "w") as f:
            f.write(f"{len(e_pwr)}\n")
            for e, T, p, cs2, s in zip(e_pwr, T_pwr, P_pwr, cs2_pwr, s_pwr):
                f.write(f"{e*MeV4_to_GeV_fm3} {T*MeV_to_GeV} {p*MeV4_to_GeV_fm3} {cs2} {s*(1/hbarc**3)}\n")

        T_w, e_w, P_w, cs2_w, s_w = make_monotonic_eos(
            T_total, e_w_file, P_w_file, cs2_w_file, s_w_file
        )
        with open(output_dir / f"eos_w_s{sample_idx}_l{l}_s{sigma}.dat", "w") as f:
            f.write(f"{len(e_w)}\n")
            for e, T, p, cs2, s in zip(e_w, T_w, P_w, cs2_w, s_w):
                f.write(f"{e*MeV4_to_GeV_fm3} {T*MeV_to_GeV} {p*MeV4_to_GeV_fm3} {cs2} {s*(1/hbarc**3)}\n")

        T_hrg, e_hrg, P_hrg, cs2_hrg, s_hrg = make_monotonic_eos(
            T_total, e_hrg_file, P_hrg_file, cs2_hrg_file, s_hrg_file
        )
        with open(output_dir / f"eos_hrg_s{sample_idx}_l{l}_s{sigma}.dat", "w") as f:
            f.write(f"{len(e_hrg)}\n")
            for e, T, p, cs2, s in zip(e_hrg, T_hrg, P_hrg, cs2_hrg, s_hrg):
                f.write(f"{e*MeV4_to_GeV_fm3} {T*MeV_to_GeV} {p*MeV4_to_GeV_fm3} {cs2} {s*(1/hbarc**3)}\n") 
    
        break

In [26]:
for i in range(0, n_samples):
    save_constrained_gp_files(sample_idx=i, l=400, sigma=15) 

/home/luizaperin/codes/gp/extrapolation_weighted.py:146: RuntimeWarning: invalid value encountered in divide
  s_total = np.where(T_total != 0, (P_total + e_total) / T_total, 0.0)


## Saving to files (unconstrained gp)

In [51]:
output_dir = Path(f"./eosDataFiles/Tsw_{T_sw}/unconstrained/l{l}_s{sigma}")
output_dir.mkdir(parents=True, exist_ok=True)

In [52]:
# for unconstrained GP 

def save_unconstrained_gp_files(sample_idx, l, sigma, mean):
    while True:
        cs2_sample_file = sampler.generate_sample_unconstrained(l=l, sigma=sigma, cte_mean=mean)
        cs2_interp_file = interp1d(T_total_coarse, cs2_sample_file) 

        s_up_file = entropy_up(s_sw, T_sw, cs2_interp_file, T_point01_up)
        P_up_file = pressure_up(P_sw, T_sw, T_point01_up, s_up_file)
        e_up_file = energy_up(s_up_file, P_up_file, T_point01_up)

        #pwr extrap 
        thermo_pwrl_file = extrap_pwrl.extrapolate_powerlaw(
          T_down=T_point001_down, 
          T_up=T_point01_up, 
          T_total=T_total, 
          cs2_interp=cs2_interp_file, 
          s_up_interp=s_up_file, 
          P_up_interp=P_up_file, 
        )

        P_pwrl_file = thermo_pwrl_file['pressure']
        e_pwrl_file = thermo_pwrl_file['energy'] 
        s_pwrl_file = thermo_pwrl_file['entropy']
        cs2_pwrl_file = thermo_pwrl_file['cs2']

        #weighted extrap 
        thermo_w_file = extrap_w.extrapolate_weighted(
          P_down=P_hrg_to110[:idx_90],
          P_up = P_up_file(T_point01_up),
          e_down=e_hrg_to110[:idx_90],
          e_up = e_up_file(T_point01_up),
          T_up = T_point01_up,
          T_total=T_total, 
          cs2_interp=cs2_interp_file, 
          n_transition=len_transition_unc,
          endpointIdx = idx_120
        )    

        P_w_file = thermo_w_file['pressure']
        e_w_file = thermo_w_file['energy'] 
        s_w_file = thermo_w_file['entropy']
        cs2_w_file = thermo_w_file['cs2']
        adjusted_transition_P = thermo_w_file["adjusted_transition_P"]
        adjusted_transition_e = thermo_w_file["adjusted_transition_e"]

        cs2_region = thermo_w_file['cs2'][idx_110:idx_120]

        if adjusted_transition_P != adjusted_transition_e:
           print("Rejected sample: transition width adjusted")
           continue

        if np.any((cs2_region < 0) | (cs2_region > 1.0001)):
           print("Rejected sample: cs2 out of bounds in transition region")
           continue

        #hrg to 150 vectors
        thermo_hrg_file = extrap_hrg.extrapolate_hrg(
          P_down=P_hrgtoTsw,
          P_up = P_up_file,
          e_down=e_hrgtoTsw,
          e_up=e_up_file,
          s_down=s_hrgtoTsw,
          s_up=s_up_file,
          T_total = T_total,
          cs2_interp=cs2_interp_file,
          idx_Tsw = idx_Tsw,
        )   

        P_hrg_file = thermo_hrg_file['pressure']
        e_hrg_file = thermo_hrg_file['energy'] 
        s_hrg_file = thermo_hrg_file['entropy']
        cs2_hrg_file = thermo_hrg_file['cs2']

        T_pwr, e_pwr, P_pwr, cs2_pwr, s_pwr = make_monotonic_eos(
            T_total, e_pwrl_file, P_pwrl_file, cs2_pwrl_file, s_pwrl_file
        )
        with open(output_dir / f"eos_pwr_s{sample_idx}_l{l}_s{sigma}.dat", "w") as f:
           f.write(f"{len(e_pwr)}\n")
           for e, T, p, cs2, s in zip(e_pwr, T_pwr, P_pwr, cs2_pwr, s_pwr):
               f.write(f"{e*MeV4_to_GeV_fm3} {T*MeV_to_GeV} {p*MeV4_to_GeV_fm3} {cs2} {s*(1/hbarc**3)}\n")

        T_w, e_w, P_w, cs2_w, s_w = make_monotonic_eos(
            T_total, e_w_file, P_w_file, cs2_w_file, s_w_file
        )
        with open(output_dir / f"eos_w_s{sample_idx}_l{l}_s{sigma}.dat", "w") as f:
           f.write(f"{len(e_w)}\n")
           for e, T, p, cs2, s in zip(e_w, T_w, P_w, cs2_w, s_w):
               f.write(f"{e*MeV4_to_GeV_fm3} {T*MeV_to_GeV} {p*MeV4_to_GeV_fm3} {cs2} {s*(1/hbarc**3)}\n")

        T_hrg, e_hrg, P_hrg, cs2_hrg, s_hrg = make_monotonic_eos(
            T_total, e_hrg_file, P_hrg_file, cs2_hrg_file, s_hrg_file
        )
        with open(output_dir / f"eos_hrg_s{sample_idx}_l{l}_s{sigma}.dat", "w") as f:
            f.write(f"{len(e_hrg)}\n")
            for e, T, p, cs2, s in zip(e_hrg, T_hrg, P_hrg, cs2_hrg, s_hrg):
                f.write(f"{e*MeV4_to_GeV_fm3} {T*MeV_to_GeV} {p*MeV4_to_GeV_fm3} {cs2} {s*(1/hbarc**3)}\n") 
    
        print("accepted sample saved")
        break

In [53]:
for i in range(0, n_samples):
     save_unconstrained_gp_files(sample_idx=i, l=400, sigma=15, mean=0.3) 

accepted sample saved
accepted sample saved
